# Production-Ready California Housing Valuation Platform

### Objective
This notebook develops a robust real estate valuation model. Rather than manipulating matrices using manual scripting loops, everything is bound into a unified Scikit-Learn Pipeline asset to prevent data leakage and guarantee reproducible real-time web inference.

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from google.colab import files

# Ingest data from a dependable, immutable public source
data_url = "https://raw.githubusercontent.com/ageron/handson-ml/master/datasets/housing/housing.csv"
raw_housing_data = pd.read_csv(data_url)
raw_housing_data.to_csv("housing.csv",index=False)
files.download('housing.csv')
# Separate features from target variable early to preserve baseline structure
X = raw_housing_data.drop(['median_house_value'], axis=1)
y = raw_housing_data['median_house_value']

# Isolate testing data before applying any transformers to simulate a true production environment
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Ingest data from a dependable, immutable public source
data_url = "https://raw.githubusercontent.com/ageron/handson-ml/master/datasets/housing/housing.csv"
raw_housing_data = pd.read_csv(data_url)

# Separate features from target variable early to preserve baseline structure
X = raw_housing_data.drop(['median_house_value'], axis=1)
y = raw_housing_data['median_house_value']

# Isolate testing data before applying any transformers to simulate a true production environment
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 1. Custom Feature Engineering & Data Imputation
To make our model production-compatible, we construct a custom transformer class. This class uses fit-time data from the training set (like the median bedroom count) to handle missing values safely when a user submits single-row requests on the live webpage.

In [4]:
from sklearn.base import BaseEstimator, TransformerMixin

class CaliforniaHousingTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.median_bedrooms_ = None

    def fit(self, X, y=None):
        # Identify the median only within the training set to prevent structural data leakage
        self.median_bedrooms_ = X['total_bedrooms'].median()
        return self

    def transform(self, X):
        X_out = pd.DataFrame(X).copy()

        # Resolve missing entries using our training anchor values
        X_out['total_bedrooms'] = X_out['total_bedrooms'].fillna(self.median_bedrooms_)

        # Calculate engineered metrics dynamically during both batch training and real-time inference
        X_out['rooms_per_household'] = X_out['total_rooms'] / X_out['households']
        X_out['bedrooms_per_room'] = X_out['total_bedrooms'] / X_out['total_rooms']
        X_out['population_per_household'] = X_out['population'] / X_out['households']

        # Strip away original broad aggregates that add noise to the regressor
        return X_out.drop(columns=['total_rooms', 'total_bedrooms', 'population', 'households'])

## 2. Preprocessing Matrix Mapping & Pipeline Integration
Now, we define our numerical scaling and categorical encoding branches, packing them tightly with our model into a final operational blueprint.

In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor  # Upgraded learning engine
import joblib
from google.colab import files

# Identify structural tracking paths
num_cols = ['longitude', 'latitude', 'housing_median_age', 'median_income',
            'rooms_per_household', 'bedrooms_per_room', 'population_per_household']
cat_cols = ['ocean_proximity']

# Package parallel processing steps
preprocessing_gate = ColumnTransformer(transformers=[
    ('numerical_scaling', StandardScaler(), num_cols),
    ('categorical_encoding', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
])

# Assemble the final consolidated executable architecture with XGBoost
housing_production_pipeline = Pipeline(steps=[
    ('custom_engineering', CaliforniaHousingTransformer()), # Retaining your custom feature step intact
    ('preprocessing_matrices', preprocessing_gate),
    ('regressor_engine', XGBRegressor(
        n_estimators=150,      # Total sequential gradient boosted trees
        max_depth=6,           # Shallow tree paths keep the final file size ultra-low
        learning_rate=0.08,    # Step size shrinkage factor to prevent overfitting
        subsample=0.8,         # Row ratio sample per training iteration
        colsample_bytree=0.8,  # Column ratio sample when splitting trees
        random_state=42,
        n_jobs=-1              # Maximize multi-threaded CPU compilation cores
    ))
])

# Fit and execute validation analysis
print("Fitting the end-to-end production XGBoost pipeline...")
housing_production_pipeline.fit(X_train, y_train)

validation_r2 = housing_production_pipeline.score(X_test, y_test)
print(f"\nModel Verified successfully! R-squared Score: {validation_r2:.4f}")

# Serialize and export the unified binary asset with native zlib compression
output_filename = 'housing_pipeline.pkl'
joblib.dump(housing_production_pipeline, output_filename)
print(f"✅ Success! Optimized model serialized")

# Trigger cloud environment download hook
files.download(output_filename)

Fitting the end-to-end production XGBoost pipeline...

Model Verified successfully! R-squared Score: 0.8219
✅ Success! Optimized model serialized


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>